In [ ]:
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica"],
    "font.size": 11,
    "axes.linewidth": 1,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": 1,
    "ytick.major.width": 1,
})

In [ ]:
input_file_path = Path(
    "../../Results/3.2_SimulatedYield/"
    "morris_sensitivity_summary_maize_TH_left_0p6_var_20260716.pkl"
)

output_directory = Path("../../fig")

histogram_file_path = (
    output_directory / "hist_morris_yield_maize_TH_left_0p6_var.svg"
)

sensitivity_plot_file_path = (
    output_directory / "morris_mu_sigma_maize_TH_left_0p6_var.svg"
)

In [ ]:
with input_file_path.open("rb") as file_handle:
    loaded_data = pickle.load(file_handle)

Si_list = loaded_data["indices"]
model_outputs = loaded_data["outputs"]
problem_definition = loaded_data["problem"]

In [ ]:
long_term_mean_yield = np.nanmean(model_outputs, axis=1)

print(
    f"Range of mean simulated yield on all simulations (t/ha): "
    f"{long_term_mean_yield.min():.2f}~"
    f"{long_term_mean_yield.max():.2f}"
)

filtered_outputs = np.where(model_outputs > 0.01, model_outputs, np.nan)
long_term_mean_yield_filtered = np.nanmean(filtered_outputs, axis=1)

print(
    f"Range of mean simulated yield on filtered simulations (t/ha): "
    f"{long_term_mean_yield_filtered.min():.2f}~"
    f"{long_term_mean_yield_filtered.max():.2f}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))

# Bars
ax.hist(
    long_term_mean_yield,
    bins=30,
    alpha=0.5,
    label="All simulations",
    edgecolor="black"
)
ax.hist(long_term_mean_yield_filtered,
        bins=30,
        alpha=0.5,
        label="Filtered simulations",
        edgecolor="black"
       )

# Labels
ax.set_xlabel("Long-term mean yield (t/ha)")
ax.set_ylabel("Frequency")
ax.legend()

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(histogram_file_path, bbox_inches="tight", dpi=300)

In [ ]:
morris_statistics = {}

for metric in ["mu_star", "sigma"]:
    stacked = np.ma.array([Si[metric] for Si in Si_list])
    morris_statistics[f"mean_{metric}"] = stacked.mean(axis=0)
    morris_statistics[f"std_{metric}"] = stacked.std(axis=0)

mean_mu_star = morris_statistics["mean_mu_star"]
std_mu_star = morris_statistics["std_mu_star"]
mean_sigma = morris_statistics["mean_sigma"]
std_sigma = morris_statistics["std_sigma"]

In [ ]:
top_variables = sorted(
    zip(
        problem_definition["names"],
        mean_mu_star,
        mean_sigma,
    ),
    key=lambda x: x[1],  # sort by mean_mu_star
    reverse=True,
)[:4]

print("Top 4 variables by mean μ*:")
for rank, (parameter_name, mu_star, sigma) in enumerate(
    top_variables, start=1
):
    print(
        f"{rank}. {parameter_name:<8} "
        f"μ* = {mu_star:.3f}, σ = {sigma:.3f}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))

ax.errorbar(
    mean_mu_star,
    mean_sigma,
    xerr=std_mu_star,
    yerr=std_sigma,
    fmt="o",
    color="black",
    ecolor="gray",
    capsize=5,
    elinewidth=1.5,
)

for parameter_name, mu_star, sigma in zip(
    problem_definition["names"],
    mean_mu_star,
    mean_sigma,
):
    ax.annotate(
        parameter_name,
        (mu_star, sigma),
        xytext=(5, 5),
        textcoords="offset points",
    )

interpretation_text = (
    "High μ*: important factor\n"
    "Low μ*: weak influence\n\n"
    "High σ: nonlinear / interaction\n"
    "Low σ: stable linear effect"
)

ax.text(
    0.05,
    0.95,
    interpretation_text,
    transform=ax.transAxes,
    fontsize=9,
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)

ax.set_xlabel(r"Mean $\mu^*$")
ax.set_ylabel(r"Mean $\sigma$")

plt.tight_layout()
plt.savefig(sensitivity_plot_file_path, bbox_inches="tight", dpi=300)